# colony_mech

> Phase 18, Stage 4 (`specs/multiagent/phase18/stage4.md`, FR-A23): the measures behind the guard advantage's decline with team size, from a replayed board. `lieTiming`: the share of lies on a guard-held clue, contest rate and median lag and lie-built answers per lie (by guard-held or not), lie-built answers per affected agent, honest openers per lied subtask; `flagMeasures`: failures per lie, the share of failures on true values, lie-built answers after a failure was visible; `guardLoad`: guard posts per turn and guard-held clues never posted by a guard. `stage4Columns` adds them to Phase 18's columns; `runStage4Cell` runs a Stage 4 cell; `reusedRows4` replays reference cells from the cache ($0) with the new columns; `replayCheck` is the regression gate (every earlier live cell through the cache-only client).

In [ ]:
#| default_exp colony_mech

In [ ]:
#| hide
from nbdev.showdoc import *

## Measures

In [ ]:
#| export
import numpy as np

from explRL.colony3 import _same
from explRL.colony_board import failures
from explRL.colony_trace import _before, colonyColumns, contaminatedBy, falseValues, replay, traceCell
from explRL.liar_trace import cacheOnlyClient
from explRL.llm_runs import LLMConfig, cellRow, runLLMCell

NAN = float('nan')
_mean = lambda xs: float(np.mean(xs)) if len(xs) else NAN
_median = lambda xs: float(np.median(xs)) if len(xs) else NAN


def _uses(board, answers, nLiar, fv):
    "[(submission, [false values it was built on])] for the honest submissions."
    subs = [s for s in board.submits if s[1] >= nLiar]
    return [(s, contaminatedBy(board, s, answers.get((float(s[0]), s[1])), fv)) for s in subs]


def lieTiming(board, answers, nLiar, guards):
    """Per false value: guard-held (a guard slot holds the clue), the first honest contest after it (an honest post of
    another value of that clue), lie-built honest wrong answers, honest openers of its subtask; summarised by guard holding."""
    G, fv = set(range(nLiar, nLiar + guards)), falseValues(board)
    con = _uses(board, answers, nLiar, fv)
    opened = {}
    for row in board.ledger:
        if row[3] == 'travel' and row[0] >= nLiar:
            opened.setdefault(row[4], set()).add(row[0])
    per = {True: [], False: []}
    for v, (t0, p0) in fv.items():
        k, j, r = v
        cont = min(((s[0], s[1]) for s in board.shareLog if s[2] == k and s[3] == j and s[4] != r and s[1] >= nLiar
                    and (s[0], s[1]) > (t0, p0)), default=None)
        per[bool(board.tasks[k].holders[j] & G)].append((cont[0] - t0 if cont else None, sum(v in c for _, c in con),
                                                         len(opened.get(k, ()))))
    users = {}
    for s, c in con:
        if c:
            users[s[1]] = users.get(s[1], 0) + 1
    out = dict(guardHeldLies=len(per[True]) / len(fv) if fv else NAN, usesPerUser=_mean(list(users.values())),
               openersPerLie=_mean([o for g in per.values() for _, _, o in g]))
    for g, name in ((True, 'Guarded'), (False, 'Unguarded')):
        lags = [x for x, _, _ in per[g] if x is not None]
        out['contestRate' + name] = len(lags) / len(per[g]) if per[g] else NAN
        out['contestLag' + name] = _median(lags)
        out['usesPerLie' + name] = _mean([u for _, u, _ in per[g]])
    return out


def flagMeasures(board, answers, nLiar):
    "Failures per false value, the share of failures on true values, and lie-built answers made after a failure on that value."
    if not getattr(board, 'failFlag', False):
        return dict(flagsPerLie=NAN, flagsOnTrue=NAN, usesAfterFlag=NAN)
    fl, fv, first = failures(board), falseValues(board), {}
    for t, i, k, x in board.failed:                      # in time order: the first failure that agrees with each value
        s = board.tasks[k]
        for j, hs in enumerate(s.holders):
            if i not in hs:
                first.setdefault((k, j, x % s.task.primes[j]), (t, i))
    tot = sum(fl.values())
    onTrue = sum(n for (k, j, r), n in fl.items() if board.tasks[k].task.clue(j)[1] == r)
    built = [(s, v) for s, c in _uses(board, answers, nLiar, fv) for v in c]
    after = sum(v in first and _before(first[v], s[0], s[1]) for s, v in built)
    return dict(flagsPerLie=sum(fl.get(v, 0) for v in fv) / len(fv) if fv else NAN,
                flagsOnTrue=onTrue / tot if tot else NAN, usesAfterFlag=after / len(built) if built else NAN)


def guardLoad(board, nLiar, guards):
    "Guard posts per guard turn, and the share of guard-held clues of opened subtasks that no guard holding them posted."
    G = set(range(nLiar, nLiar + guards))
    if not G:
        return dict(guardPostsPerTurn=NAN, guardUnposted=NAN)
    turns = sum(row[0] in G for row in board.ledger)
    posted = {(s[2], s[3], s[1]) for s in board.shareLog if s[1] in G}
    opened = {row[4] for row in board.ledger if row[3] == 'travel'}
    held = [(k, j, hs & G) for k in opened for j, hs in enumerate(board.tasks[k].holders) if hs & G]
    return dict(guardPostsPerTurn=sum(s[1] in G for s in board.shareLog) / turns if turns else NAN,
                guardUnposted=_mean([not any((k, j, g) in posted for g in gs) for k, j, gs in held]))

## Rows, cells and replays

In [ ]:
#| export
def stage4Columns(cfg, rp):
    "Phase 18's columns of a replay plus FR-A23's measures and input tokens per call."
    g, row = cfg.world.get('guards', 0), colonyColumns(rp)
    return {**row, **lieTiming(rp.board, rp.answers, rp.nLiar, g), **flagMeasures(rp.board, rp.answers, rp.nLiar),
            **guardLoad(rp.board, rp.nLiar, g),
            'inTokPerCall': row['inputTokens'] / row['calls'] if row.get('calls') else NAN}


def runStage4(cfg, seed, client):
    "runClue's signature for runLLMCell: the replay's row with Stage 4's columns, and its turn log."
    rp = replay(cfg, seed, client)
    return [stage4Columns(cfg, rp)], rp.log


def runStage4Cell(cell):
    "For runWorkload (picklable): a Stage 4 cell (numpy cells through `replay`) with Stage 4's columns."
    cfg = LLMConfig.fromDict(cell['cfg'])
    if cell['runner'] == 'numpy':
        return [cellRow(cell, cfg, stage4Columns(cfg, replay(cfg, cell['seed'])))]
    return runLLMCell(cell, run=runStage4)


def _key(r):
    return (r['study'], r['level'], int(float(r['N'])), int(float(r['seed'])))


def _diff(ref, row):
    return [k for k in ref if k in row and not k.startswith('world.') and not _same(ref[k], row[k])]


def reusedRows4(client, prior, cells):
    """The reference cells replayed through client(cfg) (the cache-only client: $0) with Stage 4's columns; every column
    of the matching prior row (same study, level, N and seed) must be unchanged."""
    idx, out = {_key(r): r for r in prior}, []
    for c in cells:
        cfg = LLMConfig.fromDict(c['cfg'])
        row = cellRow(c, cfg, stage4Columns(cfg, replay(cfg, c['seed'], client(cfg))))
        bad = _diff(idx[_key(c)], row)
        assert not bad, f"replay of {_key(c)} differs from its saved row in {bad[:5]}"
        out.append(row)
    return out


def replayCheck(cells, rows, cache):
    """The regression gate: every cell with a saved row replayed through the cache-only client (a miss raises CacheMiss,
    nothing is spent) -> dict(checked, mismatches [(key, columns)])."""
    idx, bad, n = {_key(r): r for r in rows}, [], 0
    for c in cells:
        if _key(c) in idx:
            cfg = LLMConfig.fromDict(c['cfg'])
            d = _diff(idx[_key(c)], traceCell(c, cacheOnlyClient(cfg, cache)))
            n += 1
            if d:
                bad.append((_key(c), d[:5]))
    return dict(checked=n, mismatches=bad)

## Tests

In [ ]:
from types import SimpleNamespace

from explRL.clueboard import ClueBoardEnv
from explRL.colony_board import ColonyBoardEnv

# a hand-made board: liar 0 lies on clue 1 of subtask 7 (guard 2 holds it) and on clue 0 (nobody honest guards it)
class Task:
    primes = (2, 3)
    def clue(self, j):
        return self.primes[j], (1, 2)[j]                              # x = 5
b = SimpleNamespace(nAgents=5, failFlag=False,
                    tasks={7: SimpleNamespace(task=Task(), holders=[frozenset({3, 4}), frozenset({2, 4})])},
                    shareLog=[(1.0, 0, 7, 1, 0, False, True),         # lie on clue 1
                              (1.0, 0, 7, 0, 0, False, True),         # lie on clue 0
                              (2.0, 2, 7, 1, 2, True, False)],        # guard 2 contests clue 1, one turn later
                    submits=[(1.0, 3, 7, None, 'wrong'), (3.0, 3, 7, None, 'wrong'), (3.0, 4, 7, None, 'wrong')],
                    ledger=[(3, 0.0, 1.0, 'travel', 7), (4, 0.0, 1.0, 'travel', 7), (2, 1.0, 1.0, 'stay', 7)])
ans = {(1.0, 3): 3, (3.0, 3): 0, (3.0, 4): 0}                         # 3 = x mod 3 = 0 (lie 1); 0 = both lies
t = lieTiming(b, ans, 1, 2)
assert t['guardHeldLies'] == 0.5 and t['contestRateGuarded'] == 1.0 and t['contestLagGuarded'] == 1.0
assert t['contestRateUnguarded'] == 0.0 and t['usesPerLieUnguarded'] == 2.0
assert t['openersPerLie'] == 2.0
g = guardLoad(b, 1, 2)
assert g['guardPostsPerTurn'] == 1.0 and g['guardUnposted'] == 0.0
assert all(v != v for v in flagMeasures(b, ans, 1).values())          # no failFlag: NaN
assert all(v != v for v in guardLoad(b, 1, 0).values())

In [ ]:
# failure measures on a live ColonyBoardEnv: a lie, a wrong answer built on it, then a second answer after the flag
kw = dict(m=2, r=2, R=30, value=(1, 2, 3), life=30, penalty=0.5)
f = ColonyBoardEnv(11, 8, failFlag=True, **kw)
k = f.open[0]
s = f.tasks[k]
p0, p1 = s.task.primes
i = next(iter(s.holders[0] - s.holders[1]))
liar = next(a for a in range(8) if a not in s.holders[0] | s.holders[1])
own0 = s.task.clue(0)[1]
lie1 = next(r for r in range(p1) if r != s.task.clue(1)[1] and any(x % p0 == own0 and x % p1 == r for x in range(30)))
x = next(x for x in range(30) if x % p0 == own0 and x % p1 == lie1)
f.apply(liar, ('share', (k, 1, lie1)), 0.0)
f.apply(i, ('open', k), 1.0)
f.apply(i, ('submit', x), 2.0)
f.apply(i, ('submit', x), 3.0)
m = flagMeasures(f, {(2.0, i): x, (3.0, i): x}, 0)
assert m['flagsOnTrue'] == 0.0 and m['flagsPerLie'] == 2.0 and m['usesAfterFlag'] == 0.5

In [ ]:
# Stage 4 options end to end (AC-13): replay equals the runners (numpy, and Claude with the fake client); the rules carry
# FAIL_RULE and SQUAD_RULE only when set; the board line shows failures; squads see only their squad; rollout refuses
from explRL.clue_colony import FAIL_RULE
from explRL.clue_runs import clueSetup, runClue, runClueNumpy
from explRL.clue_text import TextClueEnv
from explRL.clue_fix import fixOptions
from explRL.llm_client import LLMClient
from explRL.llm_fake import FakeClient, fakeReply

W = dict(m=2, r=2, R=30, colony=True, guards=4)
for opt in (dict(guardDeal=True), dict(failFlag=True), dict(squads=8), dict(guardDeal=True, failFlag=True, squads=8)):
    cfg = LLMConfig(task='clue', channel='blackboard', N=16, turns=10, nLiar=4, world={**W, **opt})
    assert replay(cfg, 905).rows == runClueNumpy(cfg, 905), opt
    fake = lambda: LLMClient('claude-haiku-5-5', client=FakeClient(fakeReply))
    rp = replay(cfg, 905, fake())
    ref, log = runClue(cfg, 905, fake())
    assert rp.rows == ref and rp.log == log, opt
    row = stage4Columns(cfg, rp)
    assert {'guardHeldLies', 'flagsOnTrue', 'guardPostsPerTurn', 'inTokPerCall'} <= set(row)
    if opt.get('guardDeal') and row['falseValues']:
        assert row['guardHeldLies'] == 1.0
    b, ch = clueSetup(cfg, 905)
    env = TextClueEnv(b, **fixOptions(cfg.world), nLiar=4)
    assert (FAIL_RULE in env.rules(9)) == bool(opt.get('failFlag'))
    assert ('squads of 8' in env.rules(9)) == bool(opt.get('squads'))
    if opt.get('squads'):
        obs = env.observe(9, 0.0, ch)
        mine = {k for k in b.open if b.squadOfTask[k] == 9 % 2}
        line = obs.split('\n')[1]
        assert all(f'T{k} ' in line for k in mine) and not any(f'T{k} ' in line for k in set(b.open) - mine)
        other = min(set(b.open) - mine)
        assert env.parse(dict(action='open', target=f'T{other}'), 9)[2] is False
        assert env.parse(dict(action='open', target=f'T{min(mine)}'), 9)[2] is True
        j0, r0 = next(iter(b.held(other_h := next(iter(b.tasks[other].holders[0])), other).items()))   # a clue of the other squad
        ch.post(other_h, other, (j0, b.tasks[other].task.primes[j0], r0), 0.0)
        assert f'T{other}:' not in env.observe(9, 0.0, ch).split('Clues shared on the board: ')[1].split('\n')[0]
        assert f'T{other}:' in env.observe(other_h, 0.0, ch)
        clue = f'x mod {b.tasks[other].task.primes[j0]} = {r0}'
        assert env.parse(dict(action='share', target=f'T{other}', clue=clue), 9)[2] is False
# no Stage 4 key: the plain board class and no new sentence
cfg0 = LLMConfig(task='clue', channel='blackboard', N=16, turns=10, nLiar=4, world=W)
b0, _ = clueSetup(cfg0, 905)
assert type(b0) is ClueBoardEnv and FAIL_RULE not in TextClueEnv(b0, **fixOptions(W), nLiar=4).rules(9)

In [ ]:
# the failure label on the colony line, and knownColony skipping a failed value
from explRL.clue_agents import ClueBoard
from explRL.clue_colony import colonyLine, knownColony, tally

kw = dict(m=2, r=2, R=30, value=(1, 2, 3), life=30, penalty=0.5)    # self-contained: the docs build runs cells with imports alone
f2 = ColonyBoardEnv(11, 8, failFlag=True, **kw)
ch = ClueBoard()
k = f2.open[0]
s2 = f2.tasks[k]
p0, p1 = s2.task.primes
i2 = next(iter(s2.holders[0] - s2.holders[1]))
liar = next(a for a in range(8) if a not in s2.holders[0] | s2.holders[1] and a != i2)
own0, true1 = s2.task.clue(0)[1], s2.task.clue(1)[1]
lie1 = next(r for r in range(p1) if r != true1 and any(x % p0 == own0 and x % p1 == r for x in range(30)))
for a, r in ((liar, lie1),):
    f2.apply(a, ('share', (k, 1, r)), 0.0)
    ch.post(a, k, (1, p1, r), 0.0)
viewer = next(a for a in range(8) if a not in s2.holders[1] and a not in (liar, i2))
assert knownColony(viewer, f2, ch)[k].get(1) == lie1
f2.apply(i2, ('open', k), 1.0)
f2.apply(i2, ('submit', next(x for x in range(30) if x % p0 == own0 and x % p1 == lie1)), 2.0)
assert 'in 1 failed answer)' in colonyLine(f2, tally(ch, f2), 2, failures(f2))
assert 1 not in knownColony(viewer, f2, ch)[k]

In [ ]:
# rollout refuses the Stage 4 keys
from explRL.clue_roll import rollout

try:
    rollout(LLMConfig(task='clue', channel='blackboard', N=16, turns=4, nLiar=4, world={**W, 'squads': 8}), [1], {})
    raise AssertionError('rollout should refuse')
except ValueError as e:
    assert 'Stage 4' in str(e)

In [ ]:
#| llm
# the measures reproduce the README's replay diagnosis on the 64-agent run's cached guard boards (pooled over 8 boards)
from pathlib import Path

from explRL.bench_grid import readCSV
from explRL.colony64 import stage64Grid

root = Path('..')
cache, runs = root / 'results/multiagent/llm/cache', root / 'results/multiagent/colony/stage64/runs.csv'
if runs.exists() and cache.exists():
    cells = [c for c in stage64Grid() if c['level'] == 'colonyGuards|independent']
    rows = reusedRows4(lambda cfg: cacheOnlyClient(cfg, cache), readCSV(runs), cells)
    nf = [r['falseValues'] for r in rows]
    held = sum(r['guardHeldLies'] * n for r, n in zip(rows, nf)) / sum(nf)
    ung = sum(r['usesPerLieUnguarded'] * (1 - r['guardHeldLies']) * n for r, n in zip(rows, nf)) / sum((1 - r['guardHeldLies']) * n for r, n in zip(rows, nf))
    assert abs(held - 0.375) < 0.01 and abs(ung - 1.49) < 0.01, (held, ung)
    print(f'guard-held lies {held:.3f}, unguarded uses per lie {ung:.3f}')
else:
    print('skipped: no 64-agent run or cache')